# Property Follow-up CRM — Automated Build & Release Notebook

This Google Colab notebook automates the build pipeline for **Property Follow-up CRM**:
1. Mounts Google Drive (`MyDrive/PropertyCRM/`)
2. Clones the source repository into the Colab high-speed local filesystem (`/content/build_property_crm/`)
3. Configures Android SDK command-line tools & accepts licenses
4. Runs unit tests (`testDebugUnitTest`)
5. Compiles debug APK (`assembleDebug`)
6. Computes SHA-256 checksum
7. Copies artifacts and reports to Google Drive
8. Generates release archive zip

In [ ]:
# Step 1: Mount Google Drive
from google.colab import drive
import os, subprocess, shutil, hashlib

drive.mount('/content/drive')

# Set up Drive directory structure
drive_base = '/content/drive/MyDrive/PropertyCRM'
dirs = ['SourceArchives', 'APKBuilds', 'TestReports', 'ReleaseNotes', 'DatabaseBackups']
for d in dirs:
    os.makedirs(os.path.join(drive_base, d), exist_ok=True)

print('Verified Google Drive Directory Structure:')
!ls -la /content/drive/MyDrive/PropertyCRM

In [ ]:
# Step 2: Set up Android SDK commandline tools
android_home = '/content/android-sdk'
os.environ['ANDROID_HOME'] = android_home
os.environ['ANDROID_SDK_ROOT'] = android_home

if not os.path.exists(os.path.join(android_home, 'cmdline-tools/latest/bin/sdkmanager')):
    print('Installing Android commandline tools...')
    os.makedirs('/content/android-sdk/cmdline-tools', exist_ok=True)
    !curl -fsSL https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip -o /content/cmdline-tools.zip
    !unzip -q /content/cmdline-tools.zip -d /content/android-sdk/cmdline-tools/
    !mv /content/android-sdk/cmdline-tools/cmdline-tools /content/android-sdk/cmdline-tools/latest
    !rm /content/cmdline-tools.zip

os.environ['PATH'] = f"{android_home}/cmdline-tools/latest/bin:{android_home}/platform-tools:{os.environ['PATH']}"

# Accept licenses and install platform 34 and build tools
!yes | sdkmanager --licenses > /dev/null
!sdkmanager "platforms;android-34" "build-tools;34.0.0" > /dev/null
print('Android SDK installation complete!')

In [ ]:
# Step 3: Clone or Copy Project to Local Filesystem
build_dir = '/content/build_property_crm'
if os.path.exists(build_dir):
    shutil.rmtree(build_dir)

repo_url = 'https://github.com/Vikash-45/PropertyCRM.git'
!git clone $repo_url $build_dir

os.chdir(build_dir)
!chmod +x gradlew
print('Project successfully cloned to:', build_dir)

In [ ]:
# Step 4: Run Unit Tests
%cd /content/build_property_crm
!./gradlew testDebugUnitTest --stacktrace

In [ ]:
# Step 5: Build Debug APK
%cd /content/build_property_crm
!./gradlew assembleDebug --stacktrace

In [ ]:
# Step 6: Verify APK and compute SHA-256 Checksum
import datetime
apk_path = '/content/build_property_crm/app/build/outputs/apk/debug/app-debug.apk'
assert os.path.exists(apk_path), f"APK not found at {apk_path}"

with open(apk_path, 'rb') as f:
    sha256 = hashlib.sha256(f.read()).hexdigest()

timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
dest_apk = f'/content/drive/MyDrive/PropertyCRM/APKBuilds/PropertyCRM_debug_{timestamp}.apk'
shutil.copy2(apk_path, dest_apk)
shutil.copy2(apk_path, '/content/drive/MyDrive/PropertyCRM/APKBuilds/PropertyCRM_latest.apk')

checksum_file = f'/content/drive/MyDrive/PropertyCRM/APKBuilds/checksum_{timestamp}.txt'
with open(checksum_file, 'w') as f:
    f.write(f"{sha256}  PropertyCRM_debug_{timestamp}.apk\n")

# Copy Test Reports
test_reports_src = '/content/build_property_crm/app/build/reports/tests/testDebugUnitTest'
if os.path.exists(test_reports_src):
    dest_reports = f'/content/drive/MyDrive/PropertyCRM/TestReports/test_report_{timestamp}'
    shutil.copytree(test_reports_src, dest_reports, dirs_exist_ok=True)

# Create Source Archive
source_archive = f'/content/drive/MyDrive/PropertyCRM/SourceArchives/PropertyCRM_Source_{timestamp}.tar.gz'
!tar --exclude='.git' --exclude='.gradle' --exclude='build' -czf $source_archive -C /content/build_property_crm .

print('='*50)
print('BUILD SUCCESSFUL!')
print('APK Size:', os.path.getsize(dest_apk) / (1024*1024), 'MB')
print('SHA-256 Checksum:', sha256)
print('Saved to Drive:', dest_apk)
print('='*50)